# Efficient Fine-Tuning & Continual Learning for Language Models (LoRA vs. Full FT)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/laapalhares/lora-continual-learning/blob/main/phase1_experiments.ipynb)
[![Python 3.10+](https://img.shields.io/badge/python-3.10+-blue.svg)](https://www.python.org/downloads/)
[![PyTorch](https://img.shields.io/badge/PyTorch-2.0+-ee4c2c.svg)](https://pytorch.org/)

**Author:** Larissa Palhares | Universidade Estadual de Campinas (UNICAMP)  

---

### Overview
Fine-tuning pre-trained models involves trade-offs between computational cost and knowledge retention. This experiment compares **Full Fine-Tuning** and **LoRA** across two aspects:
1. **Parameter Efficiency:** Evaluating accuracy and training time across LoRA ranks ($r \in [4, 8, 16, 32]$) with constant scaling ($\alpha / r = 2.0$) on **AG News** (4 classes).
2. **Catastrophic Forgetting:** Measuring retention drop on Task A after adapting the model to **IMDB** (sentiment classification, 2 classes).


## 1. Setup & Environment


In [ ]:
# Install dependencies if running in Google Colab
import sys
import os

if 'google.colab' in sys.modules:
    !pip install -U transformers peft accelerate datasets evaluate scikit-learn seaborn -q
    if not os.path.exists('src'):
        !git clone https://github.com/laapalhares/lora-continual-learning.git temp_repo
        !cp -r temp_repo/src ./src
        !rm -rf temp_repo

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import ExperimentConfig
from src.utils import set_seed, count_parameters
from src.data import get_tokenizer, load_and_preprocess_task_a, load_and_preprocess_task_b
from src.training import run_full_fine_tuning, run_lora_tradeoff_experiment, select_optimal_lora
from src.continual import run_catastrophic_forgetting_experiment

# Config and reproducibility
config = ExperimentConfig(epochs=3, batch_size=16, seed=42)
set_seed(config.seed)

print(f"Device in use: {config.device.upper()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


## 2. Dataset Preparation (AG News & IMDB)

Tokenize datasets and shuffle with a fixed seed before taking subsets to ensure balanced classes.


In [ ]:
tokenizer = get_tokenizer(config.model_id)

print("Loading Task A (AG News)...")
train_a, test_a = load_and_preprocess_task_a(config, tokenizer)

print("Loading Task B (IMDB)...")
train_b, test_b = load_and_preprocess_task_b(config, tokenizer)

print(f"Task A (AG News): Train = {len(train_a)} | Test = {len(test_a)}")
print(f"Task B (IMDB):    Train = {len(train_b)} | Test = {len(test_b)}")

# Check label balance for Task B
sample_labels_b = [example['labels'].item() if hasattr(example['labels'], 'item') else example['labels'] for example in test_b]
pos_count = sum(1 for l in sample_labels_b if l == 1)
neg_count = sum(1 for l in sample_labels_b if l == 0)
print(f"Task B Test Balance: {neg_count} Negative (0), {pos_count} Positive (1)")


## 3. Baseline: Full Fine-Tuning

Full Fine-Tuning updates all 66.95M parameters, serving as our reference baseline for accuracy, training time, and catastrophic forgetting.


In [ ]:
ft_metrics, model_full = run_full_fine_tuning(train_a, test_a, config)
full_trainable_params = ft_metrics["Trainable Parameters"]

df_full = pd.DataFrame([ft_metrics])
display(df_full)


## 4. Experiment 1: LoRA Parameter Efficiency

LoRA factorizes weight updates into low-rank matrices ($\Delta W = B \cdot A$). We test ranks $r \in [4, 8, 16, 32]$ with scaling $\alpha / r = 2.0$ to compare accuracy against parameter footprint.


In [ ]:
df_lora, trained_lora_models = run_lora_tradeoff_experiment(
    train_a, test_a, config, full_trainable_params
)

# Combine full fine-tuning with LoRA results
df_tradeoff = pd.concat([df_full, df_lora], ignore_index=True)
display(df_tradeoff)


### 4.1. Plotting Efficiency vs Accuracy


In [ ]:
sns.set_theme(style="whitegrid")
plt.figure(figsize=(10, 5))

full_acc = df_tradeoff[df_tradeoff["Method"] == "Full Fine-Tuning"]["Accuracy"].iloc[0]

scatter = plt.scatter(
    df_tradeoff["% Trainable"],
    df_tradeoff["Accuracy"],
    s=140,
    c=df_tradeoff["Time (s)"],
    cmap="viridis",
    edgecolor="black",
    linewidth=1.2,
    zorder=3
)
cbar = plt.colorbar(scatter)
cbar.set_label("Training Time (seconds)", fontsize=11)

for _, row in df_tradeoff.iterrows():
    label = "Full FT" if row["Method"] == "Full Fine-Tuning" else f"LoRA r={row['r']}"
    offset_x = 1.0 if row["% Trainable"] < 50 else -8.0
    plt.annotate(label, (row["% Trainable"] + offset_x, row["Accuracy"]), fontsize=10, fontweight="bold")

plt.axhline(full_acc, color="crimson", linestyle="--", linewidth=1.5, label=f"Full FT Baseline ({full_acc:.4f})")
plt.title("Accuracy vs. Trainable Parameters (Task A: AG News)", fontsize=13, fontweight="bold")
plt.xlabel("Trainable Parameters (% of Full Model)", fontsize=11)
plt.ylabel("Accuracy", fontsize=11)
plt.ylim(0.82, 0.92)
plt.legend(loc="lower right")
plt.tight_layout()

# Save table and plot to outputs/
os.makedirs("outputs", exist_ok=True)
df_tradeoff.to_csv("outputs/results_tradeoff.csv", index=False)
plt.savefig("outputs/tradeoff_frontier.png", dpi=300, bbox_inches="tight")
plt.show()


### 4.2. Selecting Best LoRA Rank

We pick the smallest rank achieving within 0.5% of peak LoRA accuracy to use in the continual learning experiment.


In [ ]:
optimal_r = select_optimal_lora(df_lora)
optimal_lora_model = trained_lora_models[optimal_r]


## 5. Experiment 2: Catastrophic Forgetting

We evaluate retention on Task A (AG News) after fine-tuning on Task B (IMDB):
1. Record initial accuracy on Task A.
2. Replace classifier head with a 2-class head and train on Task B.
3. Restore original Task A head.
4. Re-evaluate on Task A to measure accuracy drop:
$$\text{Loss (\%)} = \frac{\text{Acc}_{\text{initial}} - \text{Acc}_{\text{final}}}{\text{Acc}_{\text{initial}}} \times 100$$


In [ ]:
cf_models = {
    f"LoRA (r={optimal_r})": optimal_lora_model,
    "Full Fine-Tuning": model_full
}
initial_accs = {
    f"LoRA (r={optimal_r})": df_lora[df_lora["r"] == optimal_r]["Accuracy"].iloc[0],
    "Full Fine-Tuning": ft_metrics["Accuracy"]
}

df_cf = run_catastrophic_forgetting_experiment(
    models_dict=cf_models,
    initial_accuracies_task_a=initial_accs,
    train_dataset_b=train_b,
    test_dataset_b=test_b,
    test_dataset_a=test_a,
    config=config
)

display(df_cf)


### 5.1. Plotting Catastrophic Forgetting


In [ ]:
plt.figure(figsize=(7, 5))
colors = ["#2ecc71" if "LoRA" in m else "#e74c3c" for m in df_cf["Method"]]
bars = plt.bar(df_cf["Method"], df_cf["Loss (%)"], color=colors, edgecolor="black", width=0.45, zorder=3)

for bar in bars:
    yval = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2.0,
        yval + 0.15,
        f"{yval:.2f}%",
        ha="center",
        va="bottom",
        fontsize=11,
        fontweight="bold"
    )

plt.title("Catastrophic Forgetting (Accuracy Drop on Task A)", fontsize=13, fontweight="bold")
plt.ylabel("Accuracy Drop (%)", fontsize=11)
plt.ylim(0, max(df_cf["Loss (%)"]) + 1.5)
plt.tight_layout()

# Save table and plot to outputs/
os.makedirs("outputs", exist_ok=True)
df_cf.to_csv("outputs/results_catastrophic_forgetting.csv", index=False)
plt.savefig("outputs/catastrophic_forgetting.png", dpi=300, bbox_inches="tight")
plt.show()


## 6. Findings & Conclusion

1. **Parameter Efficiency:** LoRA ($r=4$) reaches 97.3% of Full Fine-Tuning accuracy with only 1.00% of parameters (and up to 98.6% at $r=32$), reducing training time by ~30%.
2. **Catastrophic Forgetting:** LoRA showed lower accuracy degradation on Task A after adaptation to Task B (2.73% relative loss vs. 3.60% for Full Fine-Tuning). Freezing pre-trained backbone weights ($W_0$) preserves general language representations and limits representation drift.
